# 🖥️ Computer-Use Agent with Skills

*Notebook by [Kacper Łukawski](https://www.linkedin.com/in/kacperlukawski/)*

In this notebook, we build a fully local agent that uses a **skill** to change how it reports back, and a custom `bash` tool to actually use the machine it runs on.

Haystack's [`Agent`](https://docs.haystack.deepset.ai/docs/agents) can be given **skills** - folders of instructions the agent reads on demand, in the same `SKILL.md` format used by Claude Code and Codex. A skill teaches the agent how to do something. It doesn't let the agent do anything by itself: skills teach, tools do.

By the end you'll have a fully local agent that:
1. Discovers a real skill via progressive disclosure (it only sees skill names + one-line descriptions until it decides to load one).
2. Uses the actual machine it's running on (inspecting the OS, disk space, largest files) via an **approved** `bash` command.
3. Cuts its own output tokens by loading a caveman skill built for exactly that.

**Stack:** Haystack `Agent` + `SkillToolset` + a local Ollama model + a custom `bash` `Tool` + human-in-the-loop confirmation.

## Setup

Skills ship in Haystack v3, which at the time of writing is available on PyPI only as a pre-release. Install
the pinned dev version:

In [1]:
!pip install -q "haystack-ai==3.0.0.dev20260711001657" ollama-haystack


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Running a local model with Ollama

This notebook runs fully locally with [Ollama](https://ollama.com) - no API key required. See the
[Ollama integration](https://haystack.deepset.ai/integrations/ollama) for setup, then pull the
tool-capable model this notebook uses:

```bash
ollama pull gemma4:e4b-it-qat
```


In [2]:
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma4:e4b-it-qat"

## What is a Skill?

A **skill** is just a directory:

```
skills/
  caveman/
    SKILL.md     # YAML frontmatter (name, description) + markdown instructions
    README.md    # bundled reference file, read on demand
```

`SkillToolset` exposes exactly two tools to the agent, regardless of how many skills are available:

- **`load_skill(name)`** - returns a skill's full instructions, plus a manifest of any bundled files.
- **`read_skill_file(name, path)`** - reads one of those bundled files on demand.

This scales to a large skill library because of **progressive disclosure**, and it keeps the context small even with many skills: when the toolset warms up, every skill's **name and one-line description** (not its full instructions) get added to `load_skill`'s tool description. The model sees a menu of what's available, but it only pays the token cost of a skill's full instructions when it decides to use one, and only reads bundled files it actually needs via `read_skill_file`.

## Getting a real skill

Rather than write a toy skill, we use a real, community-authored one: [`caveman`](https://github.com/JuliusBrussee/caveman) (MIT-licensed, by [Julius Brussee](https://github.com/JuliusBrussee)), which makes an agent respond in an ultra-compressed, concise style while keeping technical accuracy. Its `SKILL.md` description says it "auto-triggers when token efficiency is requested" - we rely on this later to let the model decide for itself when to use it, instead of forcing it through a system prompt.

In [3]:
%%bash
git clone --depth 1 --filter=blob:none --sparse https://github.com/JuliusBrussee/caveman.git skills_cache/caveman-skill-repo
git -C skills_cache/caveman-skill-repo sparse-checkout set skills/caveman

Cloning into 'skills_cache/caveman-skill-repo'...


In [4]:
from pathlib import Path

SKILL_NAME = "caveman"
skills_dir = Path("skills_cache/caveman-skill-repo/skills")

print("Skill files:")
for path in sorted((skills_dir / SKILL_NAME).rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(skills_dir))

Skill files:
  caveman/README.md
  caveman/SKILL.md


## Giving the skill to an agent

We pass this skill to an `Agent` and watch it decide whether to load it. There's no `bash` tool yet, so we can see `load_skill` fire on its own, before we add the `bash` tool.

In [5]:
from haystack.components.agents import Agent
from haystack.dataclasses import ChatMessage
from haystack.components.generators.utils import print_streaming_chunk
from haystack.skill_stores.file_system import FileSystemSkillStore
from haystack.tools import SkillToolset
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

print("Discovered skills:")
skills_toolset = SkillToolset(FileSystemSkillStore(skills_dir))
for info in skills_toolset.skills.values():
    print(f"  - {info.name}: {info.description}")

Discovered skills:
  - caveman: Ultra-compressed communication mode. Cuts output tokens 65% (measured) by speaking like caveman while keeping full technical accuracy. Supports intensity levels: lite, full (default), ultra, wenyan-lite, wenyan-full, wenyan-ultra. Use when user says "caveman mode", "talk like caveman", "use caveman", "less tokens", "be brief", or invokes /caveman. Also auto-triggers when token efficiency is requested.


Notice the agent hasn't loaded anything yet - `skills_toolset.skills` is metadata-only, populated purely from
`SKILL.md`'s frontmatter. We ask for something that should make the agent load the skill and show its effect
directly.

In [6]:
reader_agent = Agent(
    chat_generator=OllamaChatGenerator(model=OLLAMA_MODEL, url=OLLAMA_URL),
    system_prompt=(
        "When a task matches one of your skills, actually call the "
        "`load_skill` tool first and follow its instructions exactly - "
        "never just write out what a tool call would look like as text."
    ),
    tools=[skills_toolset],
    streaming_callback=print_streaming_chunk,
)

result = reader_agent.run(
    messages=[
        ChatMessage.from_user(
            "I need my replies to be as token-efficient as possible from "
            "now on. Load whatever skill helps with that, then answer: "
            "what is Docker and why do people use containers?"
        )
    ]
)
print(result["last_message"].text)

[TOOL CALL]
Tool: load_skill 
Arguments: {"name": "caveman"}

[TOOL RESULT]
Respond terse like smart caveman. All technical substance stay. Only fluff die.

## Persistence

ACTIVE EVERY RESPONSE. No revert after many turns. No filler drift. Still active if unsure. Off only: "stop caveman" / "normal mode".

Default: **full**. Switch: `/caveman lite|full|ultra`.

## Rules

Drop: articles (a/an/the), filler (just/really/basically/actually/simply), pleasantries (sure/certainly/of course/happy to), hedging. Fragments OK. Short synonyms (big not extensive, fix not "implement a solution for"). No tool-call narration, no decorative tables/emoji, no dumping long raw error logs unless asked - quote shortest decisive line. Standard well-known tech acronyms OK (DB/API/HTTP); never invent new abbreviations (cfg/impl/req/res/fn) - tokenizer split them same as full word: zero token saved, reader still decode. Full word cheaper AND clearer. No causal arrows (→) either - own token, save nothing. Techni

## Extending with a custom bash tool

A `SkillToolset` can only **read**. To let the agent act, and actually inspect the machine it's running on, it needs a real execution tool. Haystack has no built-in shell tool by design - running arbitrary commands is inherently risky and application-specific - so we write a small one ourselves as a plain `Tool` subclass. This lets the agent runthings like `uname -a`, `df -h`, or `find . -type f` for genuine computer use, not just describing what those commands would show. Perfectly, you should swap it for a sandboxed runner (a container, or a remote executor) without touching the skills side at all. 

> ⚠️ **Safety note:** this tool runs whatever command string the model gives it, via `shell=True`. That's fine
> for a local, single-user notebook where you approve every call. Never connect an unattended version of this
> to untrusted input.

In [7]:
import asyncio
import subprocess
from typing import Any

from haystack.core.serialization import generate_qualified_class_name
from haystack.tools import Tool

_BASH_DESCRIPTION = (
    "Execute a bash command and return its combined stdout/stderr and exit code. Use this to inspect the "
    "system, run scripts, or read/write files. Never guess or make up output - always run the real command."
)
_BASH_PARAMETERS = {
    "type": "object",
    "properties": {"command": {"type": "string", "description": "The bash command to execute."}},
    "required": ["command"],
}


class BashTool(Tool):
    """A standalone Tool that executes bash commands in a subprocess, optionally scoped to a working directory."""

    def __init__(self, working_dir: str | Path | None = None, timeout: int = 60) -> None:
        self._working_dir = Path(working_dir) if working_dir is not None else None
        self._timeout = timeout
        super().__init__(
            name="bash",
            description=_BASH_DESCRIPTION,
            parameters=_BASH_PARAMETERS,
            function=self._run,
            async_function=self._run_async,
        )

    def _format(self, returncode: int, stdout: str, stderr: str) -> str:
        parts = [f"exit_code: {returncode}"]
        if stdout:
            parts.append(f"stdout:\n{stdout.rstrip()}")
        if stderr:
            parts.append(f"stderr:\n{stderr.rstrip()}")
        return "\n".join(parts)

    def _run(self, command: str) -> str:
        cwd = str(self._working_dir) if self._working_dir is not None else None
        try:
            proc = subprocess.run(command, shell=True, cwd=cwd, capture_output=True, text=True, timeout=self._timeout)
        except subprocess.TimeoutExpired:
            return f"exit_code: -1\nstderr:\nCommand timed out after {self._timeout}s."
        return self._format(proc.returncode, proc.stdout, proc.stderr)

    async def _run_async(self, command: str) -> str:
        cwd = str(self._working_dir) if self._working_dir is not None else None
        proc = await asyncio.create_subprocess_shell(
            command, cwd=cwd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE
        )
        try:
            stdout, stderr = await asyncio.wait_for(proc.communicate(), timeout=self._timeout)
        except asyncio.TimeoutError:
            proc.kill()
            return f"exit_code: -1\nstderr:\nCommand timed out after {self._timeout}s."
        return self._format(proc.returncode or 0, stdout.decode(), stderr.decode())

    def to_dict(self) -> dict[str, Any]:
        return {
            "type": generate_qualified_class_name(type(self)),
            "data": {
                "working_dir": str(self._working_dir) if self._working_dir is not None else None,
                "timeout": self._timeout,
            },
        }

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "BashTool":
        return cls(**data["data"])


bash = BashTool()
output = bash.invoke(command="whoami")
print(output)

exit_code: 0
stdout:
kacper.lukawski


## Keeping a human in the loop

Before letting the agent actually run shell commands, we require approval for every `bash` call. Haystack's human-in-the-loop support plugs in as a `before_tool` [hook](https://docs.haystack.deepset.ai/docs/agents#hooks): a `ConfirmationHook` intercepts pending tool calls and applies a **policy** through a **UI** before the call is allowed to run.

- **Policy** - `AlwaysAskPolicy()` asks every single time (there's also `AskOncePolicy`, `NeverAskPolicy`).
- **Strategy** - `BlockingConfirmationStrategy` pauses the run and waits for your answer.
- **UI** - `SimpleConsoleUI()` prints the pending call and reads your `y.; `RichConsoleUI` is a fancier terminal-only alternative.

We scope the hook to just the `"bash"` tool by name - the skill-reading tools stay unattended, since they're read-only.

In [8]:
from haystack.human_in_the_loop import (
    AlwaysAskPolicy, 
    BlockingConfirmationStrategy, 
    ConfirmationHook, 
    SimpleConsoleUI
)

confirmation_hook = ConfirmationHook(
    confirmation_strategies={
        "bash": BlockingConfirmationStrategy(
            confirmation_policy=AlwaysAskPolicy(),
            confirmation_ui=SimpleConsoleUI(),
        )
    }
)

## Putting it together: computer use, with and without the skill

We run the **exact same** computer-use task twice: once with a plain agent, once with an agent that also has the `caveman` skill available, and compare. Both agents get the same `bash` tool and the same confirmation hook, so the only variable is whether the skill is available to load.

The task is real machine inspection: OS/kernel info, disk space, the biggest files around. Approve each `bash` call with `y` + Enter.

In [9]:
INSPECTION_QUERY = (
    "Investigate this machine: get the OS/kernel info, the Python version, "
    "disk space usage, and the 5 largest files in the current directory. "
    "Report what you find."
)

### Baseline run - no skill

In [10]:
baseline_agent = Agent(
    chat_generator=OllamaChatGenerator(model=OLLAMA_MODEL, url=OLLAMA_URL),
    system_prompt=(
        "You investigate the user's machine using the `bash` tool. "
        "Always run the real commands - never guess or make up any part "
        "of the output. If a command fails or gives incomplete output, "
        "run a corrected command instead of filling gaps with placeholder "
        "or hypothetical text."
    ),
    tools=[bash],
    hooks={"before_tool": [confirmation_hook]},
    streaming_callback=print_streaming_chunk,
    max_agent_steps=20,
)

baseline_result = baseline_agent.run(messages=[
    ChatMessage.from_user(INSPECTION_QUERY),
])

[TOOL CALL]
Tool: bash 
Arguments: {"command": "uname -a && python --version && df -h && ls -lS | head -n 6"}


--- Tool Execution Request ---
Tool: bash
Description: Execute a bash command and return its combined stdout/stderr and exit code. Use this to inspect the system, run scripts, or read/write files. Never guess or make up output - always run the real command.
Arguments:
  command: uname -a && python --version && df -h && ls -lS | head -n 6
------------------------------


Confirm execution? (y=confirm / n=reject / m=modify):  y


[TOOL RESULT]
exit_code: 0
stdout:
Darwin Kacpers-MacBook-Pro.local 25.5.0 Darwin Kernel Version 25.5.0: Mon Apr 27 20:41:15 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6041 arm64
Python 3.12.13
Filesystem        Size    Used   Avail Capacity iused ifree %iused  Mounted on
/dev/disk3s3s1   926Gi    13Gi   816Gi     2%    459k  4.3G    0%   /
devfs            201Ki   201Ki     0Bi   100%     695     0  100%   /dev
/dev/disk3s6     926Gi    20Ki   816Gi     1%       0  8.6G    0%   /System/Volumes/VM
/dev/disk3s4     926Gi    17Gi   816Gi     3%    2.1k  8.6G    0%   /System/Volumes/Preboot
/dev/disk3s2     926Gi   815Mi   816Gi     1%     464  8.6G    0%   /System/Volumes/Update
/dev/disk1s2     500Mi   6.0Mi   479Mi     2%       1  4.9M    0%   /System/Volumes/xarts
/dev/disk1s1     500Mi   5.6Mi   479Mi     2%      37  4.9M    0%   /System/Volumes/iSCPreboot
/dev/disk1s3     500Mi   4.7Mi   479Mi     1%      98  4.9M    0%   /System/Volumes/Hardware
/dev/disk3s1     926Gi    78Gi 

### Skill-aware run - same task, `caveman` available

The query below adds one phrase, "make this as token-efficient as possible," that matches `caveman`'s own trigger condition. We don't tell the agent to use the skill - it decides to, because the skill's description says that's exactly when it applies. That's progressive disclosure in action.

In [11]:
skill_agent = Agent(
    chat_generator=OllamaChatGenerator(model=OLLAMA_MODEL, url=OLLAMA_URL),
    system_prompt=(
        "You investigate the user's machine using the `bash` tool. "
        "Always run the real commands - never guess or make up any part "
        "of the output. If a command fails or gives incomplete output, "
        "run a corrected command instead of filling gaps with placeholder "
        "or hypothetical text."
    ),
    tools=[skills_toolset, bash],
    hooks={"before_tool": [confirmation_hook]},
    streaming_callback=print_streaming_chunk,
    max_agent_steps=20,
)

skill_query = INSPECTION_QUERY + " Make this as token-efficient as possible."

skill_result = skill_agent.run(messages=[ChatMessage.from_user(skill_query)])

[ASSISTANT]
The user wants a summary of several system aspects:
1.  OS/Kernel info
2.  Python version
3.  Disk space usage
4.  The 5 largest files in the current directory

The constraint is to make this "as token-efficient as possible." This strongly suggests using the `caveman` skill, which reduces output tokens by speaking like a caveman while maintaining technical accuracy and supports different intensity levels (e.g., `wenyan-lite`). I should choose an appropriate level, perhaps "ultra" or rely on the default if not specified further, to be maximally token efficient in the *output*.

First, I'll load the `caveman` skill to prepare for a concise output format. Then I will chain the necessary `bash` commands.
The required information can be gathered with:
1.  OS/Kernel info: `uname -a`
2.  Python version: `python3 --version` (assuming python3 is standard, or just `python --version`)
3.  Disk space usage: `df -h /` or `du -sh ./*` depending on context; `df -h` gives overall disk usag

Confirm execution? (y=confirm / n=reject / m=modify):  y


[TOOL RESULT]
exit_code: 0
stdout:
Darwin Kacpers-MacBook-Pro.local 25.5.0 Darwin Kernel Version 25.5.0: Mon Apr 27 20:41:15 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6041 arm64
Python 3.12.13
Filesystem        Size    Used   Avail Capacity iused ifree %iused  Mounted on
/dev/disk3s3s1   926Gi    13Gi   816Gi     2%    459k  4.3G    0%   /

[TOOL CALL]
Tool: bash 
Arguments: {"command": "find . -max 1 -type f -printf \"%s %p\\n\" | sort -rn | head -n 5"}


--- Tool Execution Request ---
Tool: bash
Description: Execute a bash command and return its combined stdout/stderr and exit code. Use this to inspect the system, run scripts, or read/write files. Never guess or make up output - always run the real command.
Arguments:
  command: find . -max 1 -type f -printf "%s %p\n" | sort -rn | head -n 5
------------------------------


Confirm execution? (y=confirm / n=reject / m=modify):  y


[TOOL RESULT]
exit_code: 0
stderr:
find: -max: unknown primary or operator

[TOOL CALL]
Tool: bash 
Arguments: {"command": "find . -type f -exec du -h {} + | sort -rh | head -n 5"}


--- Tool Execution Request ---
Tool: bash
Description: Execute a bash command and return its combined stdout/stderr and exit code. Use this to inspect the system, run scripts, or read/write files. Never guess or make up output - always run the real command.
Arguments:
  command: find . -type f -exec du -h {} + | sort -rh | head -n 5
------------------------------


Confirm execution? (y=confirm / n=reject / m=modify):  y


[TOOL RESULT]
exit_code: 0
stdout:
206M	./.venv/lib/python3.14/site-packages/torch/lib/libtorch_cpu.dylib
 33M	./.venv/lib/python3.14/site-packages/cv2/cv2.abi3.so
 28M	./.venv/lib/python3.14/site-packages/torch/lib/libtorch_python.dylib
 10M	./.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
9.8M	./.venv/lib/python3.14/site-packages/cv2/.dylibs/libavcodec.61.19.101.dylib

[ASSISTANT]
Kernel: Darwin 25.5.0 arm64. Python 3.12.13. Disk `/` used 2%. Top files: `torch_cpu.dylib`, `cv2/cv2.abi3.so`, etc. Report done.



### The comparison

There are two different things worth measuring here, and they can point in different directions:

- **Final-answer tokens** (`result["messages"][-1].meta["usage"]`): tokens spent on just the reply you read. This isolates the skill's effect on style.
- **Whole-run total tokens** (`result["token_usage"]`): all LLM calls in the run, including every step spent deciding which tool to call next. This depends mostly on how many separate tool calls the model made, not on the skill. Don't treat a difference here as proof the skill worked or didn't.

In [12]:
baseline_final_tokens = baseline_result["messages"][-1].meta.get("usage", {}).get("completion_tokens", 0)
skill_final_tokens = skill_result["messages"][-1].meta.get("usage", {}).get("completion_tokens", 0)
baseline_total_tokens = baseline_result["token_usage"].get("completion_tokens", 0)
skill_total_tokens = skill_result["token_usage"].get("completion_tokens", 0)

metrics = {
    "final-answer tokens": (baseline_final_tokens, skill_final_tokens),
    "whole-run total tokens": (baseline_total_tokens, skill_total_tokens),
}
for label, (before, after) in metrics.items():
    change = f"{100 * (after - before) / before:+.0f}%" if before else "n/a"
    print(f"{label:<22} baseline {before:>4}   with skill {after:>4}   change {change:>5}")

final-answer tokens    baseline  300   with skill   59   change  -80%
whole-run total tokens baseline  334   with skill  497   change  +49%


## What's next?

- **Sandbox the bash tool.** For anything beyond a local notebook, run commands in a container or a restricted shell instead of a bare `subprocess`.
- **Combine with more tools.** `tools=[skills_toolset, bash, *other_tools]` - a `SkillToolset` composes with any other `Tool`/`Toolset`, as long as there's only one `SkillToolset` per agent.
- **Try per-run tool selection.** `agent.run(tools=[...])` lets you restrict which tools are active for a given call - handy once an agent has a larger toolbox.